# Import Libraries

In [49]:
import os
import pandas as pd
import numpy as np

# Loading Cleaned Dataset

In [17]:
customers = pd.read_csv("processed/customers_clean.csv")
orders = pd.read_csv("processed/orders_clean.csv")
order_items = pd.read_csv("processed/order_items_clean.csv")
payments = pd.read_csv("processed/payments_clean.csv")

In [32]:
customers.shape, orders.shape, order_items.shape, payments.shape

((99441, 5), (96478, 11), (98666, 4), (99440, 4))

In [31]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [59]:
df['order_purchase_timestamp'].dtype

dtype('<M8[ns]')

# Merging Customers Column with Orders

In [50]:
df = customers.merge(
    orders,
    on='customer_id',
    how='inner')

# Merging Order Items(Revenue)

In [51]:
df = df.merge(
    order_items,
    on='order_id',
    how='left')

# Merging Payments

In [52]:
df = df.merge(
    payments,
    on='order_id',
    how='left')

# Dataset Overview after merge

In [36]:
df.shape

(96478, 21)

In [37]:
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,order_estimated_delivery_date,delivery_time_days,estimated_vs_actual,order_month,total_order_value,total_items,avg_item_price,total_payment_value,max_installments,payment_types
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,2017-06-05,8.0,10.0,2017-05,146.87,1,124.99,146.87,2.0,1.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,2018-02-06,16.0,7.0,2018-01,335.48,1,289.00,335.48,8.0,1.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,2018-06-13,26.0,-2.0,2018-05,157.73,1,139.94,157.73,7.0,1.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,2018-04-10,14.0,12.0,2018-03,173.30,1,149.94,173.30,1.0,1.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,2018-08-15,11.0,5.0,2018-07,252.25,1,230.00,252.25,8.0,1.0


In [38]:
df.isnull().sum()

customer_id                       0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
order_id                          0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
delivery_time_days                8
estimated_vs_actual               8
order_month                       0
total_order_value                 0
total_items                       0
avg_item_price                    0
total_payment_value               1
max_installments                  1
payment_types                     1
dtype: int64

In [39]:
df.fillna(0, inplace=True)

C:\Users\urvas\AppData\Local\Temp\ipykernel_11520\4231983114.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df.fillna(0, inplace=True)


In [40]:
#df.isnull().sum()

# Feature Engineering – RFM(Recency, Frequency, Monetary)

### Recency (Days since last purchase)

In [53]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
last_purchase = df.groupby('customer_unique_id')['order_purchase_timestamp'].max()
recency = (snapshot_date - last_purchase).dt.days

### Frequency(Number of orders)

In [43]:
frequency = df.groupby('customer_unique_id')['order_id'].nunique()

### Monetary(Total spending)

In [44]:
monetary = df.groupby('customer_unique_id')['total_payment_value'].sum()

### RFM Table

In [54]:
rfm = pd.concat([recency, frequency, monetary], axis=1)
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
customer_unique_id,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
0000f46a3911fa3c0805444483337064,537,1,86.22
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [55]:
# Some Extra Features like mean of items, delivery_time and installments
extra_features = (          
    df.groupby('customer_unique_id')
    .agg(
        avg_delivery_time=('delivery_time_days', 'mean'),
        avg_items=('total_items', 'mean'),
        avg_installments=('max_installments', 'mean')
    )
)

In [56]:
customer_features = rfm.join(extra_features)
customer_features.fillna(0, inplace=True)
customer_features.head()

,Recency,Frequency,Monetary,avg_delivery_time,avg_items,avg_installments
customer_unique_id,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,6.0,1.0,8.0
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3.0,1.0,1.0
0000f46a3911fa3c0805444483337064,537,1,86.22,25.0,1.0,8.0
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,20.0,1.0,4.0
0004aac84e0df4da2b147fca70cf8255,288,1,196.89,13.0,1.0,6.0


In [57]:
# Saving Dataset of Data merge and Feature Engineering

In [58]:
os.makedirs("processed", exist_ok=True)
customer_features.to_csv("processed/final_customer_features.csv")